In [1]:
from torch import load, randn
import torch.nn as nn
import torch

import pandas as pd
import numpy as np
import os

from torchvision import transforms
from PIL import Image
from torchvision.datasets import ImageFolder

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [2]:
train_dir = '100-bird-species/train/'

best_epoch = pd.read_csv('training_history_augmented.csv')['val_accuracy'].argmax()
weights = load(f'aug_checkpoints/checkpoint_epoch_{best_epoch+1}.pt', map_location='cpu')['model_state_dict']

transformation = transforms.Compose([transforms.Resize((224, 224), Image.LANCZOS), transforms.ToTensor()])
train_data = ImageFolder(train_dir, transform = transformation)

In [3]:
class ImagePatcher(nn.Module):
    def __init__(self, image_size = 224, patch_size = 14, embedding_dim = 128):
        super(ImagePatcher, self).__init__()
        self.patch_size = patch_size
        self.embedding_dim = embedding_dim
        
        ##defining function that will patch the image up
        self.unfold = nn.Unfold(kernel_size = self.patch_size, stride = self.patch_size)
        
        ##defining the function that will do the linear embedding
        self.linear = nn.Linear(self.patch_size*self.patch_size*3, self.embedding_dim)
        
    def forward(self, x):
        #batch_size, c, h, w = x.shape
        
        ##patch the images
        patches = self.unfold(x).permute(0, 2, 1) # [batch size, c*h*w, num_of_patches]
        
        ##make linear embedding
        patches = self.linear(patches)
        
        return patches ## output should have dimensions [batch size, num_patches, embedding dimension]

class Attention(nn.Module):
    def __init__(self, hidden_dim, num_heads, embedding_dim = 128, dropout = 0.0):
        super().__init__()
        
        ##normalizations
        self.layer_norm_1 = nn.LayerNorm(embedding_dim)
        self.layer_norm_2 = nn.LayerNorm(embedding_dim)
        
        ##self-attention block
        self.attn = nn.MultiheadAttention(embedding_dim, num_heads, dropout, batch_first=True)
        ##who the fuck thought batch_first = False should be the default?!?!?!?!!??!?!?!?!
        ##bruhhh u stupid asf
        
        ##feed-forward function
        ##  relu activation function, the linear functions just make you go back and forth
        self.feed_forward = nn.Sequential(
            nn.Linear(in_features=embedding_dim, out_features=hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=hidden_dim, out_features=embedding_dim),
            nn.Dropout(dropout))
    
    def forward(self, x):
        #normalizing the tensor
        inp_x = self.layer_norm_1(x)
        
        #residual connection
        x = x + self.attn(inp_x, inp_x, inp_x)[0]
        
        #feed-forward
        x = x + self.feed_forward(self.layer_norm_2(x))
        
        return(x)

class VisionTransformer(nn.Module):
    def __init__(self, hidden_dim, num_heads, num_layers, 
                 num_classes = 525, image_size = 224, embedding_dim = 128, patch_size = 14, dropout = 0.0):
        
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (image_size//patch_size)**2
        
        ##calling the previously defined functions:
        self.patcher = ImagePatcher(image_size, patch_size, embedding_dim)
        self.attentions = nn.ModuleList([Attention(hidden_dim=hidden_dim, num_heads=num_heads, 
                                                   embedding_dim=embedding_dim, dropout=dropout) 
                                         for _ in range(num_layers)])
        
        ##droput
        self.dropout = nn.Dropout(dropout)
        
        ##do the classification thingy
        self.mlp_head = nn.Sequential(nn.LayerNorm(normalized_shape=embedding_dim), 
                                      nn.Linear(embedding_dim, num_classes))
        
        ##classification token
        self.cls_token = nn.Parameter(randn(1,1,embedding_dim))
        
        ##positional embedding
        self.pos_embedding = nn.Parameter(randn(1, self.num_patches + 1, embedding_dim))
        
        
    def forward(self, x):
        #data pre-processing
        x = self.patcher(x)
        batchSize, T, _ = x.shape
        
        #add classification token
        cls_token = self.cls_token.repeat(batchSize, 1, 1)
        x = torch.cat((cls_token, x), dim = 1)
        x = self.pos_embedding[:,:T+1] + x
        
        #apply transformer
        for trans in self.attentions:
            x = self.dropout(x)
            x = trans(x)
        
        #classification prediction
        cls = x[:,0]
        out = self.mlp_head(cls)
        
        return(out)

In [4]:
model = VisionTransformer(hidden_dim = 512, num_heads = 8, num_layers = 8, embedding_dim=128, dropout=0.1).to('cpu')
model.load_state_dict(weights)
model.eval();

In [30]:
transformation = transforms.Compose([transforms.Resize((224, 224), Image.LANCZOS), transforms.ToTensor()])
birds = train_data.classes

def get_probabililities(image_path):
    bird = transformation(Image.open(image_path)).unsqueeze(0)
    with torch.no_grad():
        output = model(bird)
    
    probabilities = torch.softmax(output, dim = 1).squeeze()
    return probabilities

def predicted_bird(image_path):
    probabilities = get_probabililities(image_path)
    most_likely = probabilities.argmax(dim=0).item()
    return birds[most_likely]

def get_bird_images(bird):
    return ['100-bird-species/test/' + bird + '/' + img for img in os.listdir('100-bird-species/test/' + bird)]

def get_bird_accuracy(bird):
    return sum([predicted_bird(img) == bird for img in get_bird_images(bird)])/len(get_bird_images(bird))

def get_probabilities_per_image(bird):
    images = get_bird_images(bird)
    probabilities = [get_probabililities(img).numpy() for img in images]

    df = pd.DataFrame(probabilities, columns = birds)
    df.insert(0, "bird_class", bird)
    df.insert(0, "image_path", images)

    return df

In [13]:
accuracy_array = [[b, get_bird_accuracy(b)] for b in birds ]
accuracy_df = pd.DataFrame(accuracy_array, columns=['Class', 'Accuracy'])

accuracy_df.to_csv('accuracy_per_bird.csv', index=False)

In [36]:
accuracy_for_all_images_df = pd.concat([get_probabilities_per_image(bird) for bird in birds])
accuracy_for_all_images_df.to_csv('image_predictions.csv.gz', index=False)